In [1]:
from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)

True

In [2]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [3]:
openai = OpenAI()

In [4]:
checklist = []
completed = []

In [6]:
def get_checklist_report() -> str:
    result = ""
    for index, item in enumerate(checklist):
        if completed[index]:
            result += f"Checklist #{index + 1}: [green][strike]{item}[/strike][/green]\n"
        else:
            result += f"Checklist #{index + 1}: {item}\n"
    show(result)
    return result

In [7]:
get_checklist_report()

''

In [8]:
def create_checklist(descriptions: list[str]) -> str:
    checklist.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_checklist_report()

In [10]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(checklist):
        completed[index - 1] = True
    else:
        return "No checklist at this index."
    Console().print(completion_notes)
    return get_checklist_report()

In [12]:
checklist, completed = [], []

create_checklist(["Buy groceries", "Finish week one", "Eat banana"])

Checklist #1: Buy groceries
Checklist #2: Finish week one
Checklist #3: Eat banana

'Checklist #1: Buy groceries\nChecklist #2: Finish week one\nChecklist #3: Eat banana\n'

In [13]:
mark_complete(1, "bought")

bought

Checklist #1: Buy groceries
Checklist #2: Finish week one
Checklist #3: Eat banana

'Checklist #1: [green][strike]Buy groceries[/strike][/green]\nChecklist #2: Finish week one\nChecklist #3: Eat banana\n'

In [14]:
create_checklist_json = {
    "name": "create_checklist",
    "description": "Add new checklist from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions of checklist items'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [15]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the checklist item at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the checklist item to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the checklist item in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [16]:
tools = [{"type": "function", "function": create_checklist_json},
        {"type": "function", "function": mark_complete_json}]

In [17]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [18]:
def loop(messages):
    response = openai.chat.completions.create(model="gpt-5.5", messages=messages, tools=tools)
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
        response = openai.chat.completions.create(model="gpt-5.5", messages=messages, tools=tools)
    show(response.choices[0].message.content)

In [19]:
system_message = """
You are given a problem to solve, by using your checklist tools to plan a list of steps, then carrying out each step in turn.
Now create a plan, set the checklist, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [20]:
checklist, completed = [], []
loop(messages)

Checklist #1: Identify known quantities and any missing quantity; estimate or assume a reasonable Boston–New York 
rail distance if not provided.
Checklist #2: Compute Boston train’s head start distance by 3:00 pm.
Checklist #3: Compute remaining distance between trains at 3:00 pm.
Checklist #4: Use combined closing speed to compute time after 3:00 pm until meeting.
Checklist #5: Convert elapsed time to clock time and present the answer with the stated assumption.

Known: Train A leaves Boston at 2:00 pm at 60 mph; Train B leaves New York at 3:00 pm at 80 mph toward Boston. 
Missing: Boston–New York distance. I’ll use a reasonable rail-route estimate of about 230 miles.

Checklist #1: Identify known quantities and any missing quantity; estimate or assume a reasonable Boston–New York 
rail distance if not provided.
Checklist #2: Compute Boston train’s head start distance by 3:00 pm.
Checklist #3: Compute remaining distance between trains at 3:00 pm.
Checklist #4: Use combined closing speed to compute time after 3:00 pm until meeting.
Checklist #5: Convert elapsed time to clock time and present the answer with the stated assumption.

From 2:00 pm to 3:00 pm is 1 hour, so the Boston train travels 60 mph × 1 h = 60 miles.

Checklist #1: Identify known quantities and any missing quantity; estimate or assume a reasonable Boston–New York 
rail distance if not provided.
Checklist #2: Compute Boston train’s head start distance by 3:00 pm.
Checklist #3: Compute remaining distance between trains at 3:00 pm.
Checklist #4: Use combined closing speed to compute time after 3:00 pm until meeting.
Checklist #5: Convert elapsed time to clock time and present the answer with the stated assumption.

...

Closing speed after 3:00 pm is 60 + 80 = 140 mph. Time to meet is 170 ÷ 140 = 1.2143 hours, about 1 hour 12.9 
minutes.

Checklist #1: Identify known quantities and any missing quantity; estimate or assume a reasonable Boston–New York 
rail distance if not provided.
Checklist #2: Compute Boston train’s head start distance by 3:00 pm.
Checklist #3: Compute remaining distance between trains at 3:00 pm.
Checklist #4: Use combined closing speed to compute time after 3:00 pm until meeting.
Checklist #5: Convert elapsed time to clock time and present the answer with the stated assumption.

Adding about 1 hour 12.9 minutes to 3:00 pm gives approximately 4:13 pm.

Checklist #1: Identify known quantities and any missing quantity; estimate or assume a reasonable Boston–New York 
rail distance if not provided.
Checklist #2: Compute Boston train’s head start distance by 3:00 pm.
Checklist #3: Compute remaining distance between trains at 3:00 pm.
Checklist #4: Use combined closing speed to compute time after 3:00 pm until meeting.
Checklist #5: Convert elapsed time to clock time and present the answer with the stated assumption.

Answer: Assuming Boston and New York are about 230 miles apart by rail, the trains meet at approximately 4:13 pm.

Reasoning:  
The Boston train travels 60 miles before the New York train leaves. That leaves 170 miles between them at 3:00 pm. 
Their combined speed is 140 mph, so they meet after 170 ÷ 140 ≈ 1.21 hours, or about 1 hour 13 minutes after 3:00 
pm.